# Sports Vision Tracking: Day 1 Model Benchmarking on Google Colab T4

This notebook conducts an **empirical benchmark** of lightweight YOLO candidates on the **free Google Colab NVIDIA T4 GPU**.

### Models Evaluated under Identical Conditions:
1. **YOLOv8n** (Nano: ~3.2M params)
2. **YOLOv8s** (Small: ~11.2M params)
3. **YOLO11n** (Nano: ~2.6M params)
4. **YOLO11s** (Small: ~9.4M params)

### Strict Benchmarking Protocol:
- **Football Test Clip**: Exactly 30 seconds (750 frames at 25.0 FPS) from `[1500, 2250)`.
- **Basketball Test Clip**: Exactly 30 seconds (900 frames at 30.0 FPS) from `[900, 1800)`, downscaled from 4K to 1080p for inference.
- **Identical Settings**: Same image size (640), same confidence threshold (0.25), same IoU (0.45), same 30-frame warm-up, and same FP16 half-precision on the T4 GPU.
- **Measured Outputs**: Raw Model Inference FPS, End-to-End Pipeline FPS, Peak GPU VRAM (MB), Total Person Detections, Total Ball Detections, and Zero-Ball Frames %.

## Step 1: Verify NVIDIA T4 GPU & System Environment

In [ ]:
!nvidia-smi

import torch
print("="*50)
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Active GPU: {gpu_name}")
    print(f"Total VRAM: {total_vram:.2f} GB")
    assert "T4" in gpu_name or total_vram >= 14.0, "Warning: Ensure you have selected the T4 GPU runtime in Colab!"
else:
    raise RuntimeError("CUDA not detected! In Google Colab, go to Runtime -> Change runtime type -> T4 GPU.")
print("="*50)

## Step 2: Mount Google Drive & Link Project Directory

Mount your Google Drive where the `Sports Vision Tracking` folder and the `Data/` subfolder are located.

In [ ]:
from google.colab import drive
import os

# Mount Drive
drive.mount('/content/drive')

# Define and navigate to the project directory
# Update this path if your folder is named differently or placed in a subfolder
PROJECT_PATH = "/content/drive/MyDrive/Sports Vision Tracking"

if not os.path.exists(PROJECT_PATH):
    # Alternative fallback if located directly in root of MyDrive
    alt_path = "/content/drive/MyDrive/Sports_Vision_Tracking"
    if os.path.exists(alt_path):
        PROJECT_PATH = alt_path

os.chdir(PROJECT_PATH)
print(f"Current Working Directory: {os.getcwd()}")
!ls -lh Data/

## Step 3: Install Minimal Day 1 Dependencies

In [ ]:
!pip install -q --upgrade pip
!pip install -q -r requirements.txt

import ultralytics
print(f"Ultralytics version: {ultralytics.__version__}")

## Step 4: Unit Test the Video Segment Loader

Verify that frame streaming, exact interval slicing `[start_frame, end_frame)`, and 4K-to-1080p resolution scaling function without memory leaks.

In [ ]:
import sys
sys.path.insert(0, os.getcwd())

from src.common.video_loader import VideoSegmentLoader, rescale_bbox

print("--- 1. Testing Football Loader (1080p @ 25 FPS) ---")
with VideoSegmentLoader("Data/Football.mp4", start_frame=1500, end_frame=1530) as loader:
    print(loader)
    packets = list(loader.stream_frames())
    assert len(packets) == 30, f"Expected 30 frames, got {len(packets)}"
    print(f"Decoded {len(packets)} frames successfully. Packet 0: Global Frame {packets[0].global_frame_idx}, Time {packets[0].global_timestamp_sec}s")

print("\n--- 2. Testing Basketball Loader (4K -> 1080p @ 30 FPS) ---")
with VideoSegmentLoader("Data/Basketball.mp4", start_frame=900, end_frame=930, target_resolution=(1920, 1080)) as loader:
    print(loader)
    packets = list(loader.stream_frames())
    assert len(packets) == 30, f"Expected 30 frames, got {len(packets)}"
    assert packets[0].scaled_dims == (1920, 1080)
    assert packets[0].scale_factors == (2.0, 2.0)
    print(f"Decoded {len(packets)} frames successfully. Native: {packets[0].original_dims} -> Scaled: {packets[0].scaled_dims} (Scale: {packets[0].scale_factors})")

print("\n>>> ALL VIDEO LOADER CHECKS PASSED!")

## Step 5: Execute Empirical Model Benchmark Suite

This runs all 4 models on 30-second clips of both sports on the T4 GPU and records:
- Model raw inference FPS
- End-to-end pipeline FPS
- Peak GPU VRAM allocated
- Person detections count
- Ball detections count & zero-ball frame percentage

In [ ]:
!python scripts/benchmark_models.py \
    --device cuda:0 \
    --imgsz 640 \
    --conf 0.25 \
    --iou 0.45 \
    --warmup 30 \
    --output_dir outputs/benchmarks

## Step 6: Review & Display Empirical Benchmark Results

In [ ]:
import pandas as pd

df = pd.read_csv("outputs/benchmarks/benchmark_results.csv")

print("\n" + "="*70)
print("           FOOTBALL BENCHMARK RESULTS (750 frames / 30 seconds)")
print("="*70)
fb_df = df[df["sport"] == "football"][[
    "model", "inference_fps", "pipeline_fps", "peak_vram_mb",
    "person_detections_total", "ball_detections_total", "zero_ball_frames_pct"
]]
display(fb_df)

print("\n" + "="*70)
print("          BASKETBALL BENCHMARK RESULTS (900 frames / 30 seconds)")
print("="*70)
bb_df = df[df["sport"] == "basketball"][[
    "model", "inference_fps", "pipeline_fps", "peak_vram_mb",
    "person_detections_total", "ball_detections_total", "zero_ball_frames_pct"
]]
display(bb_df)

## Step 7: Analysis & Decision Framework for Day 2

Compare the empirical results:
1. **Speed & Headroom**: Does the Small model (`yolov8s` or `yolo11s`) achieve $\ge 30$ Pipeline FPS on the T4 GPU?
2. **Ball Sensitivity**: How many more ball detections does Small capture compared to Nano?
3. **Memory Safety**: Is peak VRAM well within the 15.8 GB T4 limit (e.g. $< 3.5\text{ GB}$)?

Save the output CSV to your Google Drive for project documentation.